# UVLO Regenerative Latch - Resistor Design Notebook
### LM5176 USB-C PD -> 12 V Fan-Array Controller   *(rev 4 - single-sample build)*

Derives **every resistor from scratch** for the EN/UVLO lockout latch. Each
value is *computed* from the constraints that make the latch **set, hold, and
release** correctly; no board values are reused.

**Scope (read this first).** A single BJT sensing `VIN` smears its threshold by
the `Vbe` spread. Across the *full production box* (0.65-0.85 V part spread,
-10..+85 C) that smear (~1.5x in VIN) is about equal to the IC's own
turn-on/turn-off ratio, so a single BJT **cannot** place its set point above the
IC turn-off in both corners while still cold-starting. This build is therefore
**scoped to one board in a home environment (10-40 C, one specific 2N3904)**,
where the spread collapses to ~1.2x and the design becomes comfortable - it
even covers the IC's full turn-off tolerance band for the sample. **For
production, replace the BJT `VIN` sensor with a TL431** (sharp, temperature-stable
2.5 V threshold); see the final section for a drop-in sizing.

**Why a latch.** The LM5176 EN/UVLO comparator has no memory. If `VIN` browns
out while 12 V is still charged, force the controller off and keep it off until
the output collapses - else it restarts into a charged rail and risks prebiased
startup causing large negative current.

**The set requirement.** As `VIN` falls, the latch must set (Q2 off, Q1 on)
*above* the IC's own turn-off, in both corners, so it catches the brownout
before the IC drops out. The wide IC hysteresis (turn-on ~4.35 V, turn-off
~2.74 V) opens room for the BJT set point to land there while cold-starting off
the 5 V USB rail.

**Sub-0.7 V model.** The set/hold path (`Q3`->`Q4`->`Q3`) ignites from
*sub-threshold* leakage, so every device uses `Ic = Is*exp(Vbe/Vt)` anchored to
the datasheet point - never a hard 0.7 V cutoff.

## 1. Circuit topology (as analysed)

```
              VIN ---------+----------------------+
                           |                      |
                          R_UV2(top)             R4(top)
                           |                      |
        EN/UVLO pin .------+------.VB2P-sense.----+
            |              |          |           |
          (LM5176)       R_UV1(bot)  R3(bot)   [Q2 base = VB2P]
            |              |          |
           GND            GND        GND
                                       Q2: C->(R5)->VB3, E->GND  (NPN, anti-latch shunt)

        VOUT(12V) --+----------------+-------------------+
                    |                |                   |
                   R6           [Q4 emitter]            ...
                    |                |
      VB3 .---------+----R5----[Q2 collector]   (R5 = reset/anti-latch shunt)
          |         |
        [Q3 base]   +--R8--[Q4 collector]  (R8 = positive feedback into VB3)
          |
        [Q3 collector]--R7--[Q4 base]      (R7 = Q4 base drive)
          |
         GND
                     [Q4 collector]--R9--[Q1 base];  Q1: C->EN pin, E->GND
```

**Transistors:** `Q1,Q2,Q3 = 2N3904` (NPN), `Q4 = 2N3906` (PNP).

| Ref | Role |
|-----|------|
| `R_UV2 / R_UV1` | EN/UVLO sense - IC thresholds; `I_HYS` out of pin sets the wide hysteresis |
| `R4 / R3` | `Q2` `VIN`-sense - `Q2` turns **off** as `VIN` falls past the set point, arming the latch |
| `R6` | `VOUT`->`VB3` set pull-up | 
| `R5` | `VB3`->`Q2`-collector reset/anti-latch shunt |
| `R7` | `Q4` base drive | `R8` | positive feedback | `R9` | `Q1` EN-clamp drive |

### Regions the design must satisfy
| # | VIN | VOUT | Requirement |
|---|-----|------|-------------|
| (a) | high (run, >= 5 V) | high | **must NOT** self-latch |
| (b) | falling, **above IC turn-off** | high | **must** latch (both corners) |
| (c) | recovered high | high | once latched, **stays** latched -> bistable |
| (d) | recovered high | low  | **must** release so the IC restarts, but not too early or too much energy might flow backwards|

`Q2` is modelled as a **real exponential device driven by VIN**, so the set
point falls out of the actual circuit per corner.

In [ ]:
import sys, os
import numpy as np
import pandas as pd
from scipy.optimize import brentq
import matplotlib.pyplot as plt

for p in [".", "..", "../.."]:
    if os.path.exists(os.path.join(p, "constants.py")) and p not in sys.path:
        sys.path.insert(0, p)
try:
    import constants as C
except ModuleNotFoundError as e:
    raise ModuleNotFoundError("constants.py must sit next to this notebook.") from e

EN, DT = C.IC_LM5176, C.DESIGN_TARGETS
E_SERIES = np.sort(np.array(C.MFG.BASIC_0603_RESISTORS))

print("Loaded constants.py")
print(f"  EN threshold V_EN  = {EN.V_EN_MIN}/{EN.V_EN_NOM}/{EN.V_EN_MAX} V")
print(f"  Hysteresis   I_HYS = {EN.I_HYS_MIN*1e6:.2f}/{EN.I_HYS_NOM*1e6:.2f}/{EN.I_HYS_MAX*1e6:.2f} uA (~2:1 spread)")
print(f"  Standby      I_STBY= {EN.I_STBY_NOM*1e6:.1f}/{EN.I_STBY_MAX*1e6:.1f} uA")
print(f"  VOUT nominal {C.FAN.V_NOMINAL} V ; VIN max {C.USB.V_MAX*(1+C.USB.V_TOLERANCE):.1f} V")
print(f"  JLCPCB basic >= 100k: {[f'{v/1e3:g}k' for v in E_SERIES if v >= 100e3]}")

Loaded constants.py
  EN threshold V_EN  = 1.17/1.22/1.29 V
  Hysteresis   I_HYS = 2.15/3.15/4.25 uA (~2:1 spread)
  Standby      I_STBY= 2.0/4.0 uA
  VOUT nominal 12.0 V ; VIN max 22.0 V
  JLCPCB basic >= 100k: ['100k', '120k', '150k', '200k', '220k', '270k', '300k', '330k', '470k', '510k', '1000k', '2000k', '10000k']


## 2. Device model + two corner boxes

`Ic = Ic_ref * exp((Vbe - Vbe_eff)/(n*Vt))`, `Vt = kT/q`. `Vbe_eff` is the
effective turn-on at the 10 mA anchor and is **temperature-inclusive** - we do
*not* additionally apply -2 mV/C (that double-counts temperature and makes `Q3`
absurdly leaky).

**PRODUCTION box** (what a mass-produced unit must survive): part spread
0.65-0.85 V, -10..+85 C. The hot/cold `Vbe` smear is ~1.5x in VIN - about equal
to the IC turn-on/turn-off ratio, so a single BJT can't win (this is why
production wants a TL431).

**HOME / SINGLE-SAMPLE box** (this build): one specific 2N3904, 10-40 C. Take
typical `Vbe_eff ~ 0.66 V` with +/-0.04 V for the unknown individual part, giving
0.62-0.70 V; the smear shrinks to ~1.2x and the set point places cleanly.

| Box | corner | `Vbe_eff` | `Tc` | beta | role |
|-----|--------|-----------|------|------|------|
| HOME | **LO** (warm/leaky) | 0.62 | 40 C | 200 | sets *lowest* -> binds "above turn-off" |
| HOME | **HI** (cool/strong)| 0.70 | 10 C | 100 | sets *highest* -> binds cold-start |
| PROD | EASY | 0.65 | 85 C | 150 | reference only |
| PROD | HARD | 0.85 | -10 C | 40 | reference only |

`VCE_SAT = 0.10 V`, forced-beta `beta_F = 10`, resistor tolerance +/-1.5 %.

In [ ]:
Q_E, K_B = 1.602176634e-19, 1.380649e-23
def Vt(Tc): return K_B*(273.15+Tc)/Q_E
def ic_of_vbe(vbe, ve, Tc, ic_ref=10e-3, n=1.0): return ic_ref*np.exp((vbe-ve)/(n*Vt(Tc)))
def vbe_of_ic(ic, ve, Tc, ic_ref=10e-3, n=1.0): return ve + n*Vt(Tc)*np.log(max(ic,1e-15)/ic_ref)

class Corner:
    def __init__(s, n, ve, Tc, b): s.name, s.vbe_eff, s.Tc, s.beta = n, ve, Tc, b

# HOME / single-sample design corners
LO = Corner("warm", 0.62, 40, 200)   # leaky/warm  -> latch sets at LOWEST VIN
HI = Corner("cool", 0.70, 10, 100)   # strong/cool -> latch sets at HIGHEST VIN
# PRODUCTION reference corners
EASY = Corner("easy", 0.65,  85, 150)
HARD = Corner("hard", 0.85, -10,  40)
VCE_SAT, BETA_F, R_TOL = 0.10, 10.0, 0.015

def spread(a, b, I=30e-6): return vbe_of_ic(I,b.vbe_eff,b.Tc)/vbe_of_ic(I,a.vbe_eff,a.Tc)
print(f"Vbe set-point spread (cold/hot @30uA):  HOME = {spread(LO,HI):.3f}   PROD = {spread(EASY,HARD):.3f}")
print(f"IC turn-on/turn-off ratio (nominal)  =  {EN.V_EN_NOM*(1+510/150)-EN.I_STBY_NOM*510e3:.2f}"
      f" / {EN.V_EN_NOM*(1+510/150)-EN.I_STBY_NOM*510e3-EN.I_HYS_NOM*510e3:.2f}"
      f" = {(EN.V_EN_NOM*4.4-EN.I_STBY_NOM*510e3)/(EN.V_EN_NOM*4.4-EN.I_STBY_NOM*510e3-EN.I_HYS_NOM*510e3):.2f}")
print("--> PROD spread ~ the IC ratio (single BJT can't win); HOME spread is well under it (this build works).")

## STEP 1 - EN/UVLO sense divider (`R_UV2`, `R_UV1`) - wide hysteresis

Currents sourced out of the pin (corrected sign - `I_STBY` *lowers* the rising
threshold):

$$V_{IN,on}=V_{EN}\Big(1+\tfrac{R_{UV2}}{R_{UV1}}\Big)-I_{STBY}R_{UV2},\quad
V_{IN,off}=V_{IN,on}-I_{HYS}R_{UV2}.$$

Hysteresis `= I_HYS*R_UV2`. Target turn-on in `[3.8, 4.5] V`, hysteresis >=
1.5 V, min power. (Basics jump 510k -> 1M, so 510k -> ~1.6 V is the widest
practical value; `V_OFF_MIN`=3.6 is superseded - the *system* stays off via the
latch, not the IC UVLO.)

In [ ]:
def vin_on(R2, R1, Ven=EN.V_EN_NOM, Is=EN.I_STBY_NOM): return Ven*(1+R2/R1) - Is*R2
def vin_off(R2, R1, Ven=EN.V_EN_NOM, Is=EN.I_STBY_NOM, Ih=EN.I_HYS_NOM): return vin_on(R2,R1,Ven,Is)-Ih*R2

cand = []
for R2 in E_SERIES[(E_SERIES>=200e3)&(E_SERIES<=2e6)]:
    for R1 in E_SERIES[(E_SERIES>=50e3)&(E_SERIES<=510e3)]:
        vo, vf = vin_on(R2,R1), vin_off(R2,R1)
        if DT.V_ON_SAFE_FLOOR<=vo<=DT.V_USB_MIN and (vo-vf)>=1.5 and vf>0.5:
            cand.append((22.0**2/(R2+R1), vo-vf, R2, R1, vo, vf))
cand.sort(key=lambda t:(-round(t[1],2), t[0]))
_, hys, R_UV2, R_UV1, Von, Voff = cand[0]
print(f"Selected: R_UV2={R_UV2/1e3:g}k  R_UV1={R_UV1/1e3:g}k")
print(f"  turn-ON {Von:.3f} V | turn-OFF(nom) {Voff:.3f} V | hysteresis {hys:.3f} V")
print(f"  divider power @9V={9**2/(R_UV2+R_UV1)*1e6:.0f}uW  @22V={22**2/(R_UV2+R_UV1)*1e6:.0f}uW")

## STEP 2 - Threshold plan + IC turn-off tolerance band

`VIN_OFF_IC = Voff` (~2.74 V nominal) - the latch must set *above* this.
`V_SRC_MIN = 4.75 V` (USB-C 5 V-mode minimum) - the cold-corner set point must
stay below it so the latch can't fire as `VOUT` rises at start-up.
`VOUT_HOLD_MIN = 3.5 V`, `VOUT_REL = 2.0 V`.

Because `I_HYS` spans ~2:1, the IC's own turn-off has a wide band. With the home
box this build can place the set point above the *whole* nominal band for the
sample.

In [ ]:
VOUT_NOM = C.FAN.V_NOMINAL
VIN_MAX  = C.USB.V_MAX*(1+C.USB.V_TOLERANCE)
VIN_OFF_IC, V_SRC_MIN, VOUT_HOLD_MIN, VOUT_REL = Voff, 4.75, 3.5, 2.0
Voff_hi = vin_on(R_UV2,R_UV1,EN.V_EN_MAX) - EN.I_HYS_MIN*R_UV2
Voff_lo = vin_on(R_UV2,R_UV1,EN.V_EN_MIN) - EN.I_HYS_MAX*R_UV2
print(f"VIN_OFF_IC(nom)={VIN_OFF_IC:.2f} V   band [{Voff_lo:.2f}, {Voff_hi:.2f}] V")
print(f"Von={Von:.2f} V  V_SRC_MIN={V_SRC_MIN} V  VIN_MAX={VIN_MAX:.1f} V  VOUT_HOLD_MIN={VOUT_HOLD_MIN} V")

## STEP 3 - Core + bistability solver; size `R5`, `R6`, `R8` (home box)

KCL at `VB3`: in `= (VOUT-VB3)/R6 + min(beta*Ic3,(VOUT-0.1)/R8)`; out `= Ic3/beta
+ min((VB3-VCE_SAT)/R5, q2cap)`, where `q2cap` is `Q2`'s collector capacity (huge
when `VIN` up, ~0 when collapsed). A root is stable when `f` decreases through
zero. We size the core on the two `Q2` extremes (`q2cap=INF` on / `0` off) and
**require all 8 R5,R6,R8 tolerance corners** to pass on the home box (the leaky
0.62 V Q3 is the tight case for "no self-latch", so the search keeps the shunt
strong enough).

In [ ]:
def core_roots(VOUT, co, R5, R6, R8, q2cap):
    def kcl(v):
        ic3 = ic_of_vbe(v, co.vbe_eff, co.Tc)
        ic4 = min(co.beta*max(ic3,0.0), (VOUT-0.1)/R8)
        i_in = (VOUT-v)/R6 + (ic4 if v < VOUT-0.1 else 0.0)
        sh   = min((v-VCE_SAT)/R5, q2cap) if v > VCE_SAT else 0.0
        return i_in - (ic3/co.beta + sh)
    xs = np.linspace(0.0, max(VOUT-0.05,0.1), 3000); fs = np.array([kcl(x) for x in xs]); out=[]
    for i in np.where(np.diff(np.sign(fs))!=0)[0]:
        try:
            r = brentq(kcl, xs[i], xs[i+1], xtol=1e-9)
            out.append((r, (kcl(r+1e-4)-kcl(r-1e-4))<0, ic_of_vbe(r,co.vbe_eff,co.Tc)))
        except Exception: pass
    return out
INF = 1e3
def is_latched(V, co, R5,R6,R8, c): return any(s and ic>1e-4 for _,s,ic in core_roots(V,co,R5,R6,R8,c))
def is_off(V, co, R5,R6,R8, c):     return any(s and ic<1e-5 for _,s,ic in core_roots(V,co,R5,R6,R8,c))
def q2cap(VIN, co, R3, R4):         return ic_of_vbe(VIN*R3/(R3+R4), co.vbe_eff, co.Tc)

def core_ok(R5,R6,R8):
    return (is_off(12,LO,R5,R6,R8,INF) and (is_latched(12,HI,R5,R6,R8,0) and not is_off(12,HI,R5,R6,R8,0))
            and is_latched(12,HI,R5,R6,R8,INF) and is_latched(VOUT_HOLD_MIN,HI,R5,R6,R8,INF)
            and not is_latched(VOUT_REL,LO,R5,R6,R8,INF))
best=None
for R6c in E_SERIES[(E_SERIES>=47e3)&(E_SERIES<=470e3)]:
    for R5c in E_SERIES[(E_SERIES>=1e3)&(E_SERIES<=6.8e3)]:
        if R6c/R5c<30: continue
        for R8c in E_SERIES[(E_SERIES>=4.7e3)&(E_SERIES<=22e3)]:
            if not core_ok(R5c,R6c,R8c): continue
            if sum(core_ok(R5c*a,R6c*b,R8c*d) for a in(.985,1.015) for b in(.985,1.015) for d in(.985,1.015))<8:
                continue
            p=VOUT_NOM**2/R6c+VOUT_NOM**2/R8c
            if best is None or p<best[0]: best=(p,R5c,R6c,R8c)
_,R5,R6,R8 = best
print(f"Core: R5={R5/1e3:g}k R6={R6/1e3:g}k R8={R8/1e3:g}k  (8/8 tol on home box, latched {best[0]*1e3:.1f} mW)")

## STEP 4 - `Q2` `VIN`-sense divider (`R4` top, `R3` bottom)

Sweep `VIN` against the real solver to find the **forced-set point** per corner
(where the off-state vanishes): the **warm** corner sets lowest (must exceed
`Voff`), the **cool** corner sets highest (must stay below `V_SRC_MIN` for
cold-start). With the home spread (~1.2x) there is room to put the *worst-case*
warm set point above the IC turn-off and, in fact, above the **entire nominal
`Voff` band** for this sample, while the cool set point stays under 4.75 V. The
search maximises the worst-case warm set point, then minimises power.

In [ ]:
def forced_set(co, R3, R4):
    for V in np.arange(6.0, 0.5, -0.02):
        if not is_off(VOUT_NOM, co, R5, R6, R8, q2cap(V, co, R3, R4)): return round(V, 2)
    return None

pick=None
for R4c in E_SERIES[(E_SERIES>=200e3)&(E_SERIES<=2e6)]:
    for R3c in E_SERIES[(E_SERIES>=68e3)&(E_SERIES<=510e3)]:
        r=R3c/(R3c+R4c)
        if not (0.10<r<0.30): continue
        fh, fc = forced_set(LO,R3c,R4c), forced_set(HI,R3c,R4c)
        if fh is None or fc is None or fh<=Voff or fc>=min(Von,V_SRC_MIN): continue
        if not all(is_off(VOUT_NOM,HI,R5,R6,R8,q2cap(V_SRC_MIN,HI,R3c*a,R4c*b))
                   for a in(1-R_TOL,1+R_TOL) for b in(1-R_TOL,1+R_TOL)): continue
        wh=min(forced_set(LO,R3c*a,R4c*b) for a in(1-R_TOL,1+R_TOL) for b in(1-R_TOL,1+R_TOL))
        if wh<=Voff: continue
        score=(round(wh,2), -VIN_MAX**2/(R3c+R4c))
        if pick is None or score>pick[0]: pick=(score,R3c,R4c,fh,fc,wh)
_,R3,R4,fh,fc,wh = pick
if   wh>=Voff_hi: cover=f"the ENTIRE turn-off band [{Voff_lo:.2f},{Voff_hi:.2f}] incl. resistor tolerance"
elif fh>=Voff_hi: cover=f"the full band at nominal (worst-tol {wh:.2f} is within ~{Voff_hi-wh:.2f} V of the {Voff_hi:.2f} V high corner)"
elif fh>Voff:     cover=f"the nominal turn-off {Voff:.2f} V with {fh-Voff:.2f} V margin"
else:             cover="FAIL"
print(f"Selected: R4={R4/1e3:g}k  R3={R3/1e3:g}k  (ratio {R3/(R3+R4):.3f})")
print(f"  forced-set: WARM={fh:.2f} V (worst-tol {wh:.2f}) | COOL={fc:.2f} V")
print(f"  WARM set covers {cover}")
print(f"  cold-start: COOL set {fc:.2f} < V_SRC_MIN {V_SRC_MIN} (margin {V_SRC_MIN-fc:.2f} V)")
print(f"  divider power @22V = {VIN_MAX**2/(R3+R4)*1e6:.0f} uW")

## STEP 5 - `R7` (Q4 base drive) and `R9` (Q1 EN-clamp drive)

Saturated-stage drives, forced-beta `beta_F = 10`, 2x margin at the hold floor.
`R7` sources `Q4`'s base (`Ib4 ~ (VOUT/R8)/beta_F`); `R9` drives `Q1`, which
sinks only `Ic1 ~ VIN_MAX/R_UV2 ~ 43 uA` thanks to the high-impedance EN divider,
so `R9` can be large.

In [ ]:
ib4 = (VOUT_NOM/R8)/BETA_F
R7  = next(r for r in E_SERIES[(E_SERIES>=1e3)&(E_SERIES<=22e3)][::-1] if (VOUT_HOLD_MIN-0.9)/r > 2*ib4)
ic1 = VIN_MAX/R_UV2
R9  = next(r for r in E_SERIES[(E_SERIES>=1e3)&(E_SERIES<=470e3)][::-1] if (VOUT_HOLD_MIN-0.85)/r > 2*(ic1/BETA_F))
print(f"R7 = {R7/1e3:g}k  (Ib4={ib4*1e6:.0f}uA -> {(VOUT_HOLD_MIN-0.9)/R7/ib4:.1f}x)")
print(f"R9 = {R9/1e3:g}k  (Ic1_max={ic1*1e6:.0f}uA -> {(VOUT_HOLD_MIN-0.85)/R9/(ic1/BETA_F):.1f}x)")

## STEP 6 - Full verification (home box)

Seven region tests, the 8 core tolerance corners, and an off-state sweep across
`VIN` showing exactly where the latch sets per corner.

In [ ]:
def regions(R5,R6,R8,R3,R4):
    return [
        is_off(12,LO,R5,R6,R8,q2cap(22.0,LO,R3,R4)),         # (a) no self-latch @22V warm
        is_off(12,LO,R5,R6,R8,q2cap(V_SRC_MIN,LO,R3,R4)),    # (a') reset-stable @4.75V warm
        (forced_set(LO,R3,R4) or 0)>Voff,                    # (b) warm sets above Voff
        (forced_set(HI,R3,R4) or 0)>Voff,                    # (b') cool sets above Voff
        is_latched(12,HI,R5,R6,R8,q2cap(22.0,HI,R3,R4)),     # (c) holds latched @22V cool
        is_latched(VOUT_HOLD_MIN,HI,R5,R6,R8,INF),           # hold to 3.5V
        not is_latched(VOUT_REL,LO,R5,R6,R8,INF),            # release (VIN recovered) by 2.0V
    ]
labs=["(a) no self-latch @22V","(a') reset-stable @4.75V","(b) warm sets above Voff",
      "(b') cool sets above Voff","(c) holds latched @22V","hold to 3.5V","release by 2.0V"]
res=regions(R5,R6,R8,R3,R4)
print("1) Region tests:")
for l,x in zip(labs,res): print(f"   {'PASS' if x else 'FAIL'}  {l}")
print(f"   --> ALL PASS: {all(res)}\n")
import itertools
nt=sum(all(regions(R5*a,R6*b,R8*c,R3,R4)) for a,b,c in itertools.product([1-R_TOL,1+R_TOL],repeat=3))
print(f"2) Core tolerance corners: {nt}/8\n")
print("3) Off-state (reset) vs VIN at VOUT=12 V  [rst / SET=latch fires]:")
for co in (LO,HI):
    fsv=forced_set(co,R3,R4)
    row=[f"{V:.2f}:{'rst' if is_off(12,co,R5,R6,R8,q2cap(V,co,R3,R4)) else 'SET'}"
         for V in [22,9,5,fsv+0.1,fsv,fsv-0.2,3.0]]
    print(f"   {co.name:>4}: "+"  ".join(row))

## STEP 7 - Final BOM and operating map (single sample)

In [ ]:
bom = pd.DataFrame([
    ("R_UV2",R_UV2,"EN sense top (hysteresis)"),("R_UV1",R_UV1,"EN sense bottom"),
    ("R4",R4,"Q2 VIN-sense top"),("R3",R3,"Q2 VIN-sense bottom"),
    ("R5",R5,"anti-latch/reset shunt"),("R6",R6,"VOUT->VB3 pull-up"),
    ("R7",R7,"Q4 base drive"),("R8",R8,"positive feedback"),("R9",R9,"Q1 EN-clamp drive"),
], columns=["Ref","Ohms","Role"])
bom["Value"]=bom["Ohms"].apply(lambda x:f"{x/1e6:g}M" if x>=1e6 else f"{x/1e3:g}k")
bom["basic"]=bom["Ohms"].apply(lambda x:bool(np.any(np.isclose(E_SERIES,x))))
print(bom[["Ref","Value","Role","basic"]].to_string(index=False))
assert bom["basic"].all()
print(f"\nVIN: ON {Von:.2f} | OFF(nom) {Voff:.2f} (band {Voff_lo:.2f}-{Voff_hi:.2f}) | hys {Von-Voff:.2f} V")
print(f"     latch-set: WARM {fh:.2f} .. COOL {fc:.2f} V  (cold-start floor {V_SRC_MIN} V)")

fig,(axv,ax1)=plt.subplots(1,2,figsize=(13,4.2))
axv.axvspan(Voff_lo,Voff_hi,color="tab:red",alpha=0.12,label=f"IC turn-off band")
axv.axvspan(fh,fc,color="tab:orange",alpha=0.30,label="latch-set band (warm..cool)")
axv.axvline(Voff,color="tab:red",lw=2,label=f"Voff nom {Voff:.2f}")
axv.axvline(Von,color="tab:green",lw=2,label=f"Von {Von:.2f}")
axv.axvline(V_SRC_MIN,color="tab:blue",ls="--",lw=1.5,label=f"cold-start floor {V_SRC_MIN}")
axv.set_xlim(0,Von+1.2); axv.set_yticks([]); axv.set_xlabel("VIN (V)")
axv.set_title("VIN map (single sample)"); axv.legend(fontsize=7,loc="upper left")

Vs=np.linspace(1,12,120)
ax1.fill_between(Vs,0,[1 if is_latched(v,HI,R5,R6,R8,INF) else 0 for v in Vs],step="mid",alpha=0.35,label="cool: holds")
ax1.fill_between(Vs,0,[0 if is_latched(v,LO,R5,R6,R8,INF) else 1 for v in Vs],step="mid",alpha=0.35,label="warm: released")
ax1.axvline(VOUT_HOLD_MIN,ls="--",c="k",lw=1); ax1.axvline(VOUT_REL,ls=":",c="k",lw=1)
ax1.set_xlabel("VOUT (V)"); ax1.set_yticks([]); ax1.set_title("Hold / release map"); ax1.legend(fontsize=8)
plt.tight_layout(); plt.show()
print("\nSingle-sample design complete - all checks pass on the home box.")

## STEP 8 - Production path: replace the `Q2` BJT with a **TL431**

The single-BJT sensor only works here because the home environment shrinks the
`Vbe` smear. For production (full temperature, part-to-part spread), drop the
`Q2` divider and use a **TL431** as the `VIN` comparator:

* Tie the TL431 **REF** to a `VIN` divider `R_top/R_bot`; the cathode pulls only
  when `VIN*R_bot/(R_top+R_bot) >= V_REF` (2.495 V, **+/-0.5 % and ~30 ppm/C** -
  i.e. essentially flat over temperature, versus the ~19 % `Vbe` smear a BJT
  shows over the same box).
* Trip voltage `V_trip = V_REF*(1 + R_top/R_bot)`; the cathode (open-collector-like)
  replaces `Q2`'s collector into the `R5` node - same latch core, same `R5..R9`.
* Add the TL431's ~1-2 V minimum cathode voltage and >=1 mA bias headroom; both
  are satisfied here since the node swings to `VOUT`.

The result is a **sharp, temperature-stable** `VIN` threshold: the warm/cool set
spread collapses from ~0.7 V to a few tens of mV, so the latch set point can sit
just above the IC turn-off in *all* production corners - the thing a single BJT
cannot do. The cell below sizes the TL431 divider.

In [ ]:
V_REF = 2.495   # TL431 reference, +/-0.5%
print("TL431 VIN-sense divider (production drop-in for the Q2 stage):")
for V_trip in [3.6, 3.8, 4.0]:
    ratio = V_trip/V_REF - 1           # R_top/R_bot
    Rbot = 10e3
    Rtop_ideal = Rbot*ratio
    # snap R_top to nearest basic
    Rtop = min(E_SERIES, key=lambda r: abs(r - Rtop_ideal))
    V_act = V_REF*(1 + Rtop/Rbot)
    print(f"  trip {V_trip:.2f} V: R_top/R_bot={ratio:.3f} -> R_bot=10k, R_top~{Rtop_ideal/1e3:.1f}k "
          f"(snap {Rtop/1e3:g}k -> {V_act:.2f} V).  Threshold temp-drift ~+/-0.5% vs ~19% for a BJT.")
print("\nSame latch core (R5..R9) is reused; only the Q2 sensor changes.")